# Travel Constraint Extraction

**Hyojin Lee · Recent Areas of AI · 2026**

This notebook follows the project from the input data to the recorded experiments and a London itinerary demonstration. A Qwen2.5 Transformer extracts structured conditions from an English request. Python checks the output and applies supported edits to a reference day.

The notebook recalculates the published scores from saved predictions and calls the original planner. All cells can run without downloading models or starting training. The training section inspects the settings and saved adapter from the completed run.

The data and reference answers are AI-generated, with no human annotation review. The final results describe 100 synthetic test requests.

## Reading and running

Read the saved cell outputs directly on GitHub, or open this notebook in Jupyter from a downloaded copy of this repository. A Python kernel with `jsonschema==4.26.0` is sufficient for the executed walkthrough. The optional live demo uses the separate setup in the final section.

In [1]:
from collections import Counter
from hashlib import sha256
from html import escape
from pathlib import Path
import json
import sys

from IPython.display import HTML, display

ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "experiments/final100/protocol.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")
sys.path.insert(0, str(ROOT / "src"))

from travel_constraints.contract import validate_output
from travel_constraints.metrics import score_one, summarize
from travel_constraints.planner import apply_constraints

def read_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def read_jsonl(relative_path):
    return [json.loads(line) for line in
            (ROOT / relative_path).read_text(encoding="utf-8").splitlines()
            if line.strip()]

def show_table(headers, rows):
    header = "<tr>" + "".join(f"<th>{escape(str(value))}</th>" for value in headers) + "</tr>"
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML("<table><thead>" + header + "</thead><tbody>" + body + "</tbody></table>"))

print("Project data and implementation imports are ready.")

Project data and implementation imports are ready.


## 1 Input and target output

The model receives both an itinerary and a request. The itinerary resolves phrases such as “the gallery” and provides the original return time. The target JSON records the condition type, target, operator, value, priority and clarification flag. Walking conditions may include a unit and scope. A weather field specifies when a condition applies.

The training example below leaves the walking scope unspecified. Its reference answer therefore retains the 15-minute limit and marks the scope for clarification.

In [2]:
records = read_jsonl("data/expanded/records.jsonl")
sample = records[0]

print("Example ID:", sample["id"])
print("Request:", sample["input"]["change_request"])
show_table(
    ["Time", "Reference activity"],
    [[stop["time"], stop.get("place", stop["id"])]
     for stop in sample["input"]["reference_itinerary"]],
)
print("Target JSON")
print(json.dumps(sample["target"], indent=2, ensure_ascii=False))
assert validate_output(sample["target"]) == []

Example ID: expanded_replay_0000
Request: Set a walking time allowance of at most 15 minutes.


Time,Reference activity
09:33,The Courtauld Gallery
12:11,lunch
14:13,South Bank
15:29,Victoria and Albert Museum
20:21,hotel_return


Target JSON
{
  "constraints": [
    {
      "type": "walking",
      "target": "walking_duration",
      "operator": "<=",
      "value": 15,
      "priority": "hard",
      "needs_clarification": true,
      "condition": null,
      "unit": "minutes",
      "scope": null
    }
  ]
}


`hard` records a requirement and `soft` records a preference. An exact hotel time uses `=`, while “by 19:00” uses `<=`. If a walking limit could refer to each walk or to the whole day, the unresolved scope needs a clarification flag.

The schema defines which JSON structures are accepted. Schema validation checks structure. It does not establish that the model interpreted the request correctly.

## 2 Dataset and split

The expanded corpus has 800 training records, 100 validation records and 100 final test records. Its training split combines 280 records from the original training set with 520 new examples. The original 400-record corpus supports a separate initial comparison on 80 test requests.

The cells check the published dataset checksum and inspect its split and annotation status. Repeated wording patterns and shared intents still limit generalization even when exact request texts do not cross splits.

In [3]:
audit = read_json("data/expanded/audit.json")
protocol = read_json("experiments/final100/protocol.json")
dataset_digest = sha256((ROOT / "data/expanded/records.jsonl").read_bytes()).hexdigest()

assert dataset_digest == protocol["dataset_sha256"] == audit["dataset_sha256"]
assert len(records) == 1000
counts = Counter(record["split"] for record in records)
assert dict(counts) == {"train": 800, "validation": 100, "test": 100}

show_table(["Split", "Records"], [[split, counts[split]] for split in ("train", "validation", "test")])
show_table(
    ["Audit item", "Recorded value"],
    [
        ["Original training records replayed", audit["original_training_replay"]],
        ["New training records", audit["new_training"]],
        ["Distinct training request texts", audit["unique_requests"]["train"]],
        ["Exact request overlap across splits", audit["request_overlap_between_splits"]],
        ["Reference overlap across splits", audit["reference_overlap_between_splits"]],
        ["Human-reviewed records", audit["human_reviewed"]],
    ],
)
print("Dataset checksum matches the frozen evaluation protocol.")

Split,Records
train,800
validation,100
test,100


Audit item,Recorded value
Original training records replayed,280
New training records,520
Distinct training request texts,752
Exact request overlap across splits,0
Reference overlap across splits,0
Human-reviewed records,0


Dataset checksum matches the frozen evaluation protocol.


## 3 Transformer and LoRA training

The initial study used Qwen2.5-0.5B-Instruct without fine-tuning as a simple baseline, its LoRA version, and a 4-bit 7B model prompted with two examples.

The improvement study uses Qwen2.5-1.5B-Instruct for both adapters. LoRA updates a smaller set of parameters while keeping the pretrained weights fixed. Training continued the earlier 840-update adapter for another 1,600 updates over the 800 training records. Prompt tokens were masked in the loss so the target was the annotated JSON continuation.

The following settings come from the distributed adapter configuration and the completed CPU training manifest.

In [4]:
training_config = read_json("artifacts/qwen15_improved/adapter_config.json")
training_run = read_json("experiments/final100/training_manifest.json")
adapter_check = read_json("experiments/final100/adapter_check.json")
evidence = read_json("experiments/final100/report_evidence.json")

show_table(
    ["Training setting", "Value"],
    [
        ["Base model", training_config["model"]],
        ["Training records", counts["train"]],
        ["Additional updates", training_config["iters"]],
        ["Earlier adapter updates", evidence["earlier_primary_updates"]],
        ["Optimizer", training_config["optimizer"]],
        ["Learning rate", training_config["learning_rate"]],
        ["Batch size", training_config["batch_size"]],
        ["Seed", training_config["seed"]],
        ["LoRA rank", training_config["lora_parameters"]["rank"]],
        ["LoRA layers", training_config["num_layers"]],
        ["Maximum sequence length", training_config["max_seq_length"]],
        ["Mask prompt loss", training_config["mask_prompt"]],
        ["Device", training_run["device"]],
        ["Recorded duration in hours", f'{training_run["elapsed_seconds"] / 3600:.2f}'],
        ["Optimizer state restored", training_run["optimizer_state_restored"]],
    ],
)
assert training_run["exit_code"] == 0
assert training_run["dataset_sha256"] == dataset_digest
adapter_file = ROOT / "artifacts/qwen15_improved/adapters.safetensors"
assert sha256(adapter_file.read_bytes()).hexdigest() == adapter_check["sha256"]
assert adapter_check["all_finite"] and adapter_check["reload_exact"]
print(f'Saved adapter verified. The training record reports {adapter_check["array_count"]} finite tensors and an exact reload.')

Training setting,Value
Base model,Qwen/Qwen2.5-1.5B-Instruct
Training records,800
Additional updates,1600
Earlier adapter updates,840
Optimizer,adam
Learning rate,2e-05
Batch size,1
Seed,44
LoRA rank,8
LoRA layers,4


Saved adapter verified. The training record reports 56 finite tensors and an exact reload.


The adapter, training configuration and run manifest document an experiment completed before this notebook was assembled. The historical launcher in `src/travel_constraints/train.py` has its own defaults and review gate. The normalized configuration above records the final continuation run.

Validation exposed errors in time comparisons, required versus preferred walking caps, unresolved scope and dry-weather conditions. Development used cached validation outputs. After the validation replay, the chosen adapter and shared inference settings were fixed before the final test.

In [5]:
show_table(
    ["Validation replay on 80 requests", "Fully correct"],
    [["Earlier adapter with revised checks", "63 / 80"],
     ["Selected adapter with revised checks", "73 / 80"]],
)
show_table(
    ["Inference setting", "Value"],
    [[name, protocol[name]]
     for name in ("max_tokens", "temperature", "seed", "evidence_checks", "coverage_retries")],
)

Validation replay on 80 requests,Fully correct
Earlier adapter with revised checks,63 / 80
Selected adapter with revised checks,73 / 80


Inference setting,Value
max_tokens,768
temperature,0
seed,42
evidence_checks,True
coverage_retries,2


## 4 Initial comparison on 80 requests

The original experiment uses 80 requests from the original 400-record corpus. Each prediction is scored with the project's metric implementation.

Content F1 compares the extracted condition content. Exact match requires the entire set of conditions, priorities and clarification flags to agree with the references. An invalid JSON or schema response counts as an empty extraction. The 7B run also differs in quantization and prompting.

In [6]:
original_records = [
    record for record in read_jsonl("data/travel_requests.jsonl")
    if record["split"] == "test"
]
original_results = read_json("experiments/results/comparison.json")["metrics"]
original_rows = []

for name, label in [
    ("small_baseline", "0.5B without fine-tuning, zero-shot"),
    ("small_lora", "0.5B with LoRA, zero-shot"),
    ("large_fewshot", "7B 4-bit, two examples"),
]:
    predictions = read_jsonl(f"experiments/results/{name}.jsonl")
    recalculated = summarize(original_records, predictions)
    for key, value in recalculated.items():
        assert value == original_results[name][key], (name, key)
    original_rows.append([
        label,
        f'{recalculated["f1"]:.3f}',
        f'{round(recalculated["exact_record_accuracy"] * len(original_records))} / {len(original_records)}',
        f'{recalculated["schema_validity_rate"]:.1%}',
    ])

show_table(["Approach", "Content F1", "Fully correct", "Valid structure"], original_rows)
print("All three original comparisons match the saved metrics.")

Approach,Content F1,Fully correct,Valid structure
"0.5B without fine-tuning, zero-shot",0.000,0 / 80,0.0%
"0.5B with LoRA, zero-shot",0.795,57 / 80,97.5%
"7B 4-bit, two examples",0.782,51 / 80,93.8%


All three original comparisons match the saved metrics.


The untuned 0.5B model returned no schema-valid outputs on this test, so its F1 under this output contract is zero. The result demonstrates a format failure on these inputs. It does not establish that a model without fine-tuning has no language understanding.

The next comparison uses a different test set and a fixed 1.5B base model.

## 5 Final comparison on 100 requests

Both 1.5B adapters received the same 100 test requests and reference itineraries. Reference answers were excluded from inference inputs. The saved primary output is scored before application checks. The full system uses the same validation, fallback and targeted retry rules for both adapters.

This cell recalculates all four rows from the frozen predictions using `travel_constraints.metrics.summarize`.

In [7]:
test_records = [record for record in records if record["split"] == "test"]
assert len(test_records) == 100
assert [record["id"] for record in test_records] == protocol["test_ids"]

for filename, expected_digest in protocol["file_sha256"].items():
    actual_digest = sha256((ROOT / "experiments/final100" / filename).read_bytes()).hexdigest()
    assert actual_digest == expected_digest, filename

saved_results = read_json("experiments/final100/results.json")["test100"]
saved_predictions = {
    role: read_jsonl(f"experiments/final100/{role}_test100.jsonl")
    for role in ("baseline", "candidate")
}
recomputed = {}
result_rows = []

for stage, stage_label in [("raw_primary", "Model only"), ("full_system", "Full system")]:
    recomputed[stage] = {}
    for role, role_label in [("baseline", "Earlier adapter"), ("candidate", "Improved adapter")]:
        predictions = [
            {"id": entry["id"],
             "parsed_output": entry[stage]["parsed_output"],
             "inference_seconds": entry["inference_seconds"]}
            for entry in saved_predictions[role]
        ]
        measured = summarize(test_records, predictions)
        assert measured == saved_results[stage][role], (stage, role)
        recomputed[stage][role] = measured
        result_rows.append([
            stage_label, role_label,
            f'{measured["f1"]:.3f}',
            f'{round(measured["exact_record_accuracy"] * 100)} / 100',
            f'{measured["schema_validity_rate"]:.0%}',
        ])

show_table(["Output scored", "Adapter", "Content F1", "Fully correct", "Valid structure"], result_rows)
print("All four recalculated results match the frozen evaluation.")

Output scored,Adapter,Content F1,Fully correct,Valid structure
Model only,Earlier adapter,0.506,44 / 100,86%
Model only,Improved adapter,0.833,75 / 100,98%
Full system,Earlier adapter,0.754,61 / 100,100%
Full system,Improved adapter,0.902,82 / 100,100%


All four recalculated results match the frozen evaluation.


Additional training raised model exact match from 44 to 75 requests. With the shared checks, the full system improved from 61 to 82. The 82/100 figure includes application checks and retries. The improved model alone scored 75/100.

A single wrong or missing condition makes a request fail exact match. Schema-valid output can still contain a wrong interpretation.

## 6 Inspect actual predictions and errors

The first cell selects the first improved-system test result that is fully correct and has at least four reference conditions, to illustrate a longer request. The second inspects the recorded walking-scope failure `expanded_test_0078`.

These are saved model outputs. Displaying them does not generate or correct predictions.

In [8]:
test_by_id = {record["id"]: record for record in test_records}
candidate_by_id = {entry["id"]: entry for entry in saved_predictions["candidate"]}

illustration = next(
    entry for entry in saved_predictions["candidate"]
    if len(test_by_id[entry["id"]]["target"]["constraints"]) >= 4
    and score_one(
        test_by_id[entry["id"]]["target"],
        entry["full_system"]["parsed_output"],
    )["exact"]
)
illustration_record = test_by_id[illustration["id"]]
print("Example ID:", illustration["id"])
print("Request:", illustration_record["input"]["change_request"])
print("Saved full-system extraction")
print(json.dumps(illustration["full_system"]["parsed_output"], indent=2, ensure_ascii=False))

Example ID: expanded_test_0003
Request: Here is my updated request: Keep my hotel return at or after 19:50. Keep outdoor visits out of the itinerary whenever the weather is rainy. Today I have a total budget of 48 minutes for walking. I insist on retaining the visit to Sir John Soane's Museum.
Saved full-system extraction
{
  "constraints": [
    {
      "type": "time",
      "target": "hotel_return",
      "operator": ">=",
      "value": "19:50",
      "priority": "hard",
      "needs_clarification": false,
      "condition": null
    },
    {
      "type": "weather",
      "target": "outdoor_activity",
      "operator": "avoid",
      "value": null,
      "priority": "hard",
      "needs_clarification": false,
      "condition": {
        "weather": "rain",
        "occurs": true
      }
    },
    {
      "type": "walking",
      "target": "walking_duration",
      "operator": "<=",
      "value": 48,
      "priority": "hard",
      "needs_clarification": false,
      "condition": 

In [9]:
failure_id = "expanded_test_0078"
failure_record = test_by_id[failure_id]
failure_output = candidate_by_id[failure_id]["full_system"]["parsed_output"]

print("Request:", failure_record["input"]["change_request"])
print("Reference answer")
print(json.dumps(failure_record["target"], indent=2, ensure_ascii=False))
print("Saved system output")
print(json.dumps(failure_output, indent=2, ensure_ascii=False))
assert not score_one(failure_record["target"], failure_output)["exact"]

failed_ids = [
    entry["id"] for entry in saved_predictions["candidate"]
    if not score_one(test_by_id[entry["id"]]["target"], entry["full_system"]["parsed_output"])["exact"]
]
print("Requests that failed exact match:", len(failed_ids), "out of", len(test_records))
assert len(failed_ids) == 18

Request: Please make this adjustment: My walking-time allowance is 73 minutes; whether this applies per walk is undecided.
Reference answer
{
  "constraints": [
    {
      "type": "walking",
      "target": "walking_duration",
      "operator": "<=",
      "value": 73,
      "priority": "hard",
      "needs_clarification": true,
      "condition": null,
      "unit": "minutes",
      "scope": null
    }
  ]
}
Saved system output
{
  "constraints": [
    {
      "type": "walking",
      "target": "walking_duration",
      "operator": "<=",
      "value": 73,
      "priority": "hard",
      "needs_clarification": false,
      "condition": null,
      "unit": "minutes",
      "scope": "per_segment"
    }
  ]
}
Requests that failed exact match: 18 out of 100


The failure assigns the 73-minute cap to each walk even though the request leaves scope undecided. The reference requires unresolved scope and a clarification flag.

The improved full system matched 175 condition contents, added 21 extra conditions and missed 17 reference conditions. Ten of its 18 incorrect requests combined multiple instructions. The small family subsets do not establish a general ranking of difficulty.

## 7 Apply conditions to the London demonstration

The original planner consumes the extracted JSON. It selects the weather branch and applies supported itinerary edits. The example below replays a recorded 0.5B validation prediction from the browser demo. It is separate from the 1.5B test results above.

The request asks for lunch near the gallery, a dry-weather walk or rainy-weather book cafe, a museum visit in any weather, exact hotel arrival and less walking.

In [10]:
demo_examples = read_json("demo/saved_examples.json")
demo_example = next(item for item in demo_examples if item["id"] == "combined_03_0")
places = read_json("data/reference/london_places.json")
print("Saved example model:", demo_example["model"])
print("Request:", demo_example["input"]["change_request"])
show_table(
    ["Condition type", "Target", "Action", "Value", "Priority", "Weather"],
    [
        [condition["type"], condition["target"], condition["operator"],
         condition["value"], condition["priority"],
         json.dumps(condition["condition"])]
        for condition in demo_example["prediction"]["constraints"]
    ],
)

Saved example model: Qwen2.5-0.5B-Instruct + LoRA
Request: Put lunch near The Courtauld Gallery. Retain the walk without rain, or replace it with a book cafe if rain arrives. Keep Sir John Soane's Museum in any weather. Hotel arrival must stay exactly 20:30. Prefer the least walking overall.


Condition type,Target,Action,Value,Priority,Weather
place,lunch,near,The Courtauld Gallery,hard,null
weather,outdoor_walk,keep,None,hard,"{""weather"": ""rain"", ""occurs"": false}"
weather,outdoor_walk,replace_with,book_cafe,hard,"{""weather"": ""rain"", ""occurs"": true}"
place,place_visit,include,Sir John Soane's Museum,hard,null
time,hotel_return,=,20:30,hard,null
walking,walking_amount,minimize,None,soft,null


In [11]:
reference_plan = demo_example["input"]["reference_itinerary"]
extracted = demo_example["prediction"]

dry_result = apply_constraints(reference_plan, extracted, False, places)
rain_result = apply_constraints(reference_plan, extracted, True, places)
dry_stops = {stop["id"]: stop for stop in dry_result["itinerary"]}
rain_stops = {stop["id"]: stop for stop in rain_result["itinerary"]}

show_table(
    ["Time", "Original reference day", "Dry scenario", "Rain scenario"],
    [
        [stop["time"], stop.get("place", stop["id"]),
         dry_stops[stop["id"]].get("place", stop["id"]),
         rain_stops[stop["id"]].get("place", stop["id"])]
        for stop in reference_plan
    ],
)
print("Dry-scenario changes")
for change in dry_result["changes"]:
    print("-", change)
print("Rain-scenario changes")
for change in rain_result["changes"]:
    print("-", change)
print("Planner notes")
for note in rain_result["notes"]:
    print("-", note)

Time,Original reference day,Dry scenario,Rain scenario
10:15,The Courtauld Gallery,The Courtauld Gallery,The Courtauld Gallery
12:30,lunch,WatchHouse Somerset House,WatchHouse Somerset House
14:00,South Bank,South Bank,Book cafe (venue to choose)
16:00,Sir John Soane's Museum,Sir John Soane's Museum,Sir John Soane's Museum
20:30,hotel_return,hotel_return,hotel_return


Dry-scenario changes
- Lunch candidate: WatchHouse Somerset House near The Courtauld Gallery.
- Dry alternative: keep the planned walk.
- Kept Sir John Soane's Museum.
- Hotel arrival: 20:30 (= 20:30).
Rain-scenario changes
- Lunch candidate: WatchHouse Somerset House near The Courtauld Gallery.
- Rain alternative: replace the walk with a book cafe visit.
- Kept Sir John Soane's Museum.
- Hotel arrival: 20:30 (= 20:30).
Planner notes
- Less walking is recorded as a preference; route optimization is outside this demo.


The 14:00 South Bank walk remains in the dry scenario and becomes a book cafe visit in the rain scenario. The museum visit and 20:30 hotel return remain in both plans. A book cafe venue is still to be selected. Walking minimization is recorded as a preference, and routes and venue availability are unverified.

In the browser, **Reference example** selects the original itinerary and example request. **Saved model run** replays its stored prediction. Editing the request selects **Live local model**. **Weather context** offers a stored forecast or simulated rain/dry scenarios. Forecast mode uses the London date and hourly coverage in the stored snapshot.

Reference places are from OpenStreetMap contributors under the ODbL. Forecast data are from Open-Meteo / ECMWF under CC BY 4.0. The rain threshold is an application choice of 0.1 mm/hour.

## 8 Code and optional live use

The notebook imports the implementation files below.

| File | Role |
| --- | --- |
| [contract.py](src/travel_constraints/contract.py) and [output_schema.json](src/travel_constraints/output_schema.json) | Validate the output structure |
| [train.py](src/travel_constraints/train.py) | Historical training launcher with recorded defaults and review gate |
| [infer.py](src/travel_constraints/infer.py) and [extraction.py](src/travel_constraints/extraction.py) | Run the local Transformer and handle fallback |
| [evidence_guard.py](src/travel_constraints/evidence_guard.py) and [coverage.py](src/travel_constraints/coverage.py) | Check request consistency and handle targeted retries |
| [metrics.py](src/travel_constraints/metrics.py) | Calculate the metrics used above |
| [planner.py](src/travel_constraints/planner.py) and [weather.py](src/travel_constraints/weather.py) | Apply supported edits and weather conditions |
| [demo/app.py](demo/app.py) | Serve the browser interface |

For the saved browser demonstration, run `python demo/app.py` from the repository root, then open `http://127.0.0.1:8765`. The README contains the environment setup.

For fresh model predictions on Apple Silicon, follow the **Live model** setup in the README. It installs the pinned model runtime and runs `configure_live.py` to resolve the pinned base weights and included adapters. The notebook's completed execution uses the saved predictions above.

## 9 Limitations and authorship

All requests and reference answers were generated with AI and have no human annotation review. The corpus repeats semantic patterns, covers one city, and the completed training used one seed. Performance on real traveler requests and the feasibility of actual routes were not evaluated.

The notebook was assembled after the experiments to explain and reproduce the recorded results. Codex assisted with implementation, synthetic data, experiments, analysis and documentation. Final prediction files were not manually corrected.

### References

1. Qwen et al. [Qwen2.5 Technical Report](https://arxiv.org/abs/2412.15115).
2. Hu et al. [LoRA](https://arxiv.org/abs/2106.09685).
3. [Open-Meteo ECMWF documentation](https://open-meteo.com/en/docs/ecmwf-api).
4. [OpenStreetMap attribution and license](https://www.openstreetmap.org/copyright).